
# ?? Analyse Exploratoire des Donn?es (EDA) & Feature Engineering

```text
==========================================================================
TITRE       : Notebook d'Analyse Exploratoire (Churn Telco)
AUTEUR      : Adam Beloucif
PROJET      : ?valuation Finale Data Science Python
DATE        : F?vrier 2026
DESCRIPTION : Comprendre pourquoi les clients r?silient et pr?parer les donn?es.
==========================================================================
```

> **?? Objectif M?tier :** D?tecter les signaux faibles de r?siliation (Churn) et segmenter la base pour proposer des offres de r?tention cibl?es. 
> 
> *R?gles appliqu?es : P?dagogie, Visualisation Pro, Gestion des types et des Outliers.*



## 1. ??? Configuration & Biblioth?ques
*POURQUOI ?* : Nous installons l'environnement. On force le style visuel de `seaborn` pour que les graphiques soient professionnels et pr?ts pour un rapport. On g?re aussi les alertes (warnings) pour garder le notebook propre.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

# Configuration du style visuel pro
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
# D?sactivation des warnings inutiles pour la propret? du notebook
warnings.filterwarnings('ignore')

print("? Biblioth?ques charg?es et environnement configur?.")


??????????????????????????????????????????????????????????????????????????
## 2. ?? Chargement des Donn?es & Correction des Types
??????????????????????????????????????????????????????????????????????????

*POURQUOI ?* : Le dataset brut a souvent des d?fauts. Typiquement, dans le dataset Telco, la colonne `TotalCharges` est vue comme un objet (texte) ? cause de clients tr?s r?cents n'ayant pas encore pay? (cha?ne vide `' '`). Nous devons imp?rativement la convertir en float pour pouvoir l'analyser math?matiquement, sous peine de faire planter nos mod?les.


In [ ]:
# Chargement (Ajuster le chemin si n?cessaire)
try:
    df = pd.read_csv('data/telco_churn.csv')
    print(f"?? Dataset charg? : {df.shape[0]} lignes et {df.shape[1]} colonnes.")
except FileNotFoundError:
    print("? Erreur : Fichier introuvable. Assurez-vous d'?tre ? la racine du projet.")

# Suppression de la cl? primaire qui n'a aucune valeur pr?dictive
df.drop('customerID', axis=1, inplace=True)

# ?? Correction de la colonne 'TotalCharges' (Erreur classique de Kaggle)
# coerce force les chaines non convertibles en NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Pour les nouveaux clients (tenure = 0), ils n'ont pas encore pay?
missing_charges = df['TotalCharges'].isnull().sum()
print(f"?? Valeurs manquantes dans TotalCharges : {missing_charges} (remplac?es par 0)")
df['TotalCharges'].fillna(0, inplace=True)

# Aper?u
display(df.head())


??????????????????????????????????????????????????????????????????????????
## 3. ?? Analyse de la Variable Cible (Churn)
??????????????????????????????????????????????????????????????????????????

*POURQUOI ?* : Avant de pr?dire, il faut observer ce qu'on veut pr?dire. On cherche ici ? savoir si on a affaire ? un probl?me de classes d?s?quilibr?es (Imbalanced Data). Si la majorit? des gens restent (Non-Churn), notre mod?le pourrait "tricher" en pr?disant tout le temps "Non".


In [ ]:
plt.figure(figsize=(6, 5))
ax = sns.countplot(x='Churn', data=df, palette={'No': '#2ecc71', 'Yes': '#e74c3c'})
plt.title('Distribution de la R?siliation (Churn)', fontweight='bold')
plt.ylabel('Nombre de clients')

# Ajout des pourcentages sur les barres
total = len(df)
for p in ax.patches:
    percentage = f'{100 * p.get_height() / total:.1f}%'
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(percentage, (x, y), ha='center', va='bottom', fontweight='bold')

plt.show()

print("?? CONCLUSION M?TIER :")
print("Nous avons un d?s?quilibre mod?r? (26.5% de Churn). Lors de la mod?lisation, il faudra potentiellement")
print("utiliser l'argument `class_weight='balanced'` dans le Random Forest pour ne pas biaiser l'apprentissage.")


??????????????????????????????????????????????????????????????????????????
## 4. ?? Profil du "Churner" - Facteurs de R?siliation
??????????????????????????????????????????????????????????????????????????

*POURQUOI ?* : C'est le c?ur du m?tier. Le Machine Learning c'est bien, mais comprendre le business, c'est mieux. On croise ici le taux de d?sabonnement avec des variables cl?s comme le type de contrat et la pr?sence du support technique.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Impact du Contrat
sns.histplot(data=df, x='Contract', hue='Churn', multiple='fill', shrink=.8, ax=axes[0], palette={'No': '#bdc3c7', 'Yes': '#c0392b'})
axes[0].set_title('Taux de Churn selon le Contrat', fontweight='bold')
axes[0].set_ylabel('Proportion')

# Impact du TechSupport
sns.histplot(data=df, x='TechSupport', hue='Churn', multiple='fill', shrink=.8, ax=axes[1], palette={'No': '#bdc3c7', 'Yes': '#c0392b'})
axes[1].set_title('Taux de Churn vs Support Technique', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

print("?? INSIGHTS M?TIER :")
print("1. Danger imm?diat sur le 'Month-to-month' : Ces contrats sans engagement fuient massivement.")
print("2. L'absence de support technique aggrave le churn. Vendre l'option 'TechSupport' fid?lise fortement.")


??????????????????????????????????????????????????????????????????????????
## 5. ?? Ing?nierie des Caract?ristiques (Feature Engineering)
??????????????????????????????????????????????????????????????????????????

*POURQUOI ?* : Les algorithmes adorent l'information synth?tis?e. Une donn?e brute comme l'anciennet? en mois (`tenure`) peut ?tre bruit?e. En la regroupant par paliers ("groupes d'anciennet?"), on aide le mod?le non-supervis? (Segmentation) et supervis? (Classification) ? rep?rer des patterns plus ?vidents. 


In [ ]:
# Cr?ation des segments d'anciennet?
def group_tenure(month):
    if month <= 12: return '0-1 an'
    elif month <= 24: return '1-2 ans'
    elif month <= 48: return '2-4 ans'
    else: return '4+ ans'

df['Tenure_Group'] = df['tenure'].apply(group_tenure)

# V?rification rapide de la nouvelle variable
plt.figure(figsize=(8, 4))
sns.countplot(x='Tenure_Group', hue='Churn', data=df, order=['0-1 an', '1-2 ans', '2-4 ans', '4+ ans'], palette='dark:salmon')
plt.title('Churn par groupe d\'anciennet?', fontweight='bold')
plt.show()

print("?? Feature Engineering appliqu? avec succ?s : Cr?ation de `Tenure_Group`.")
print("La prochaine ?tape sera l'encodage de toutes ces variables pour le Machine Learning.")